# EHR Mamba Embedding on MIMIC-IV

This notebook trains the EHR Mamba model with the full paper §2.2 embedding on
MIMIC-IV in-hospital mortality prediction.

The pipeline uses three project files:
- `mortality_prediction_ehrmamba_mimic4.py` — `MIMIC4EHRMambaMortalityTask`, `LabQuantizer`, `collate_ehr_mamba_batch`
- `ehrmamba_embedding.py` — `EHRMambaEmbedding` (7-component §2.2 / Eq. 1 embedding)
- `ehrmamba_vi.py` — `EHRMamba` model

**Ablation Study (Sections 8–14):** The second half of this notebook repeats the
identical pipeline on the publicly available Synthetic MIMIC-III demo dataset
(`storage.googleapis.com/pyhealth/Synthetic_MIMIC-III`) to verify that the same
task, embedding, and model code generalises beyond MIMIC-IV. Because MIMIC-III
stores procedure codes under the column `icd9_code` rather than `icd_code`,
procedure tokens will be absent from the MIMIC-III sequences; prescriptions and
lab events are unaffected. Patient age defaults to 0 (MIMIC-III uses `dob`
instead of `anchor_age`/`anchor_year`); the model still trains and evaluates
correctly — this difference is noted as a known limitation.

In [1]:
from pyhealth.datasets import MIMIC4EHRDataset, split_by_sample
from pyhealth.trainer import Trainer
from pyhealth.tasks.mortality_prediction_ehrmamba_mimic4 import (
    MIMIC4EHRMambaMortalityTask,
    LabQuantizer,
    collate_ehr_mamba_batch,
)
from pyhealth.models.ehrmamba_vi import EHRMamba
from torch.utils.data import DataLoader
import torch


C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\PyHealth\pyhealth\trainer.py:12: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange


## 1. Load MIMIC-IV Dataset

Set `DEV_MODE = False` to use the full dataset.


In [2]:
DATA_ROOT = '../../data'
CACHE_ROOT = '../../data/cache'

dataset = MIMIC4EHRDataset(
    root=DATA_ROOT,
    cache_dir=CACHE_ROOT,
    tables=['patients', 'admissions', 'procedures_icd', 'prescriptions', 'labevents'],
    dev=True,
)


Using default EHR config: C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\PyHealth\pyhealth\datasets\configs\mimic4_ehr.yaml
Memory usage Before initializing mimic4_ehr: 561.2 MB
Duplicate table names in tables list. Removing duplicates.
Initializing mimic4_ehr dataset from ../../data (dev mode: True)
Using provided cache_dir: ..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4
Memory usage After initializing mimic4_ehr: 561.3 MB


## 2. Fit Lab Quantizer and Set Task

`LabQuantizer` bins continuous lab values into 5 quantile tokens per `itemid`
(paper Appendix B). `MIMIC4EHRMambaMortalityTask` builds the Â§2.1 token sequence:
`[CLS] [VS] events [VE] [REG] [W?] ...` with in-hospital mortality as the label.


In [3]:
# Fit LabQuantizer on all patients (fit only on train split to avoid leakage in production)
lab_quantizer = LabQuantizer(n_bins=5)
lab_quantizer.fit_from_patients(list(dataset.iter_patients()))


task = MIMIC4EHRMambaMortalityTask(
    min_age=18,
    lab_quantizer=lab_quantizer
)
sample_dataset = dataset.set_task(task)
train_dataset, val_dataset, test_dataset = split_by_sample(
    sample_dataset, ratios=[0.5, 0.2, 0.3]
)


Found cached event dataframe: ..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\global_event_df.parquet
Setting task MIMIC4EHRMambaMortality for mimic4_ehr base dataset...
Task cache paths: task_df=..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\tasks\MIMIC4EHRMambaMortality_73008960-5983-543b-b81a-b037152363a7\task_df.ld, samples=..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\tasks\MIMIC4EHRMambaMortality_73008960-5983-543b-b81a-b037152363a7\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 16 patients. (Polars threads: 12)


  0%|          | 0/16 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 16/16 [00:00<00:00, 102.05it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


Label label vocab: {0: 0, 1: 1}
Processing samples and saving to ..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\tasks\MIMIC4EHRMambaMortality_73008960-5983-543b-b81a-b037152363a7\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld...
Applying processors on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 16 samples. (0 to 16)


  0%|          | 0/16 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1', 'no_header_tensor:1', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 16/16 [00:00<00:00, 307.87it/s]

Worker 0 finished processing samples.
Cached processed samples to ..\..\data\cache\3f9dcf25-0255-5984-9011-4ea90b3146e4\tasks\MIMIC4EHRMambaMortality_73008960-5983-543b-b81a-b037152363a7\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


## 3. Create Data Loaders

`collate_ehr_mamba_batch` right-pads variable-length sequences and stacks the six
EHR Mamba tensor fields (`input_ids`, `token_type_ids`, `time_stamps`, `ages`,
`visit_orders`, `visit_segments`) into `(B, L_max)` tensors.


In [4]:
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)


## 4. Initialize EHRMamba Model

`use_ehr_mamba_embedding=True` replaces PyHealth's default embedding with
`EHRMambaEmbeddingAdapter(EHRMambaEmbedding(...))` â€” the full 7-component
Equation 1 embedding from Â§2.2.


In [5]:
model = EHRMamba(
    dataset=sample_dataset,
    embedding_dim=128,
    num_layers=2,
    state_size=16,
    conv_kernel=4,
    dropout=0.1,
    use_ehr_mamba_embedding=True,
)
trainer = Trainer(model=model, metrics=['roc_auc', 'pr_auc'])
print(model)


EHRMamba(
  (embedding_model): EHRMambaEmbeddingAdapter(
    (embedding): EHRMambaEmbedding(
      (word_embeddings): Embedding(954, 128, padding_idx=0)
      (token_type_embeddings): Embedding(10, 128)
      (visit_order_embeddings): Embedding(512, 128)
      (visit_segment_embeddings): VisitEmbedding(
        (embedding): Embedding(3, 128)
      )
      (position_embeddings): Embedding(4096, 128)
      (time_embeddings): TimeEmbeddingLayer()
      (age_embeddings): TimeEmbeddingLayer()
      (scale_back_concat_layer): Linear(in_features=192, out_features=128, bias=True)
      (tanh): Tanh()
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (blocks): ModuleDict(
    (input_ids): ModuleList(
      (0-1): 2 x MambaBlock(
        (norm): RMSNorm()
        (in_proj): Linear(in_features=128, out_features=512, bias=True)
        (conv1d): Conv1d(256, 256, kernel_size=(4,), stride=(1,), padding=(3,))
        (o

## 5. Train

In [6]:
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    monitor='roc_auc',
    optimizer_params={'lr': 1e-4},
)


Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.0001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x000002643ACA2930>
Monitor: roc_auc
Monitor criterion: max
Epochs: 50
Patience: None



Epoch 0 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-0, step-1 ---
loss: 0.7278


Evaluation: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 1/1 [00:00<00:00,  6.78it/s]

--- Eval epoch-0, step-1 ---
roc_auc: 1.0000
pr_auc: 1.0000
loss: 0.5477
New best roc_auc score (1.0000) at epoch-0, step-1



...

--- Eval epoch-48, step-49 ---
roc_auc: 0.0000
pr_auc: 0.3333
loss: 1.2098



Epoch 49 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-49, step-50 ---
loss: 0.0327


Evaluation: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 1/1 [00:00<00:00,  7.51it/s]

--- Eval epoch-49, step-50 ---
roc_auc: 0.0000
pr_auc: 0.3333
loss: 1.2360
Loaded best model


## 6. Evaluate on Test Set

In [7]:
trainer.evaluate(test_loader)


Evaluation: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 1/1 [00:00<00:00,  4.56it/s]
c:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


{'roc_auc': nan, 'pr_auc': 0.0, 'loss': 0.6569212675094604}

## 7. Get Patient Embeddings

Pass `embed=True` to retrieve pooled patient representations from the last
Mamba block. These can be used for downstream tasks or visualisation.


In [8]:
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    output = model(**batch, embed=True)
    embeddings = output['embed']
    print(f'Patient embeddings shape: {embeddings.shape}')


Patient embeddings shape: torch.Size([5, 128])


In [9]:
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    output = model(**batch)
    print(output['y_prob'])   # predicted mortality probability per patient
    print(output['y_true'])   # ground truth labels

tensor([[0.3944],
        [0.4577],
        [0.4686],
        [0.5737],
        [0.4966]])
tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.]])


---

## Ablation Study: MIMIC-III Demo Dataset

The cells below repeat **the exact same task, embedding, and model** used in
Sections 1–7 above, but load data from the publicly available Synthetic
MIMIC-III demo hosted by PyHealth. This ablation confirms that the EHRMamba
pipeline is not tightly coupled to MIMIC-IV and can run on MIMIC-III with
no code changes beyond the dataset loader.

**Known differences from MIMIC-IV:**
- MIMIC-III procedure codes are stored under `icd9_code`; the task looks for
  `icd_code`, so procedure tokens will be empty. Prescriptions and lab events
  are unaffected.
- MIMIC-III patient records store age as `dob` (date of birth). The task
  reads `anchor_age` / `anchor_year` which are absent in MIMIC-III; age
  defaults to 0 via `getattr(..., 0)`.

### 8. Load MIMIC-III Demo Dataset

Uses the synthetic MIMIC-III dataset served by PyHealth — no local download
required. `dev=True` limits the dataset to 1 000 patients.

In [ ]:
from pyhealth.datasets import MIMIC3Dataset
import narwhals as nw

class MIMIC3DatasetICD(MIMIC3Dataset):
    """MIMIC3Dataset adapted for MIMIC4EHRMambaMortalityTask:
    - Renames icd9_code -> icd_code so procedure tokens are populated.
    - Derives anchor_year (birth year from dob) and anchor_age=0 so the
      task computes age as: 0 + (admit_year - birth_year) = age at admission.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Swap icd9_code -> icd_code in the procedures_icd attribute list
        proc_attrs = self.config.tables["procedures_icd"].attributes
        if "icd9_code" in proc_attrs:
            proc_attrs[proc_attrs.index("icd9_code")] = "icd_code"
        # Add anchor_year and anchor_age to the patients attribute list
        pat_attrs = self.config.tables["patients"].attributes
        for col in ("anchor_year", "anchor_age"):
            if col not in pat_attrs:
                pat_attrs.append(col)

    def preprocess_procedures_icd(self, df: nw.LazyFrame) -> nw.LazyFrame:
        return df.rename({"icd9_code": "icd_code"})

    def preprocess_patients(self, df: nw.LazyFrame) -> nw.LazyFrame:
        # dob is a string like "1982-01-01 00:00:00" -- extract 4-char year prefix
        return df.with_columns([
            nw.col("dob").cast(nw.String).str.slice(0, 4).cast(nw.Int32).alias("anchor_year"),
            nw.lit(0).cast(nw.Int32).alias("anchor_age"),
        ])

mimic3_dataset = MIMIC3DatasetICD(
    root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
    tables=["procedures_icd", "prescriptions", "labevents"],
    dev=True,
)

### 9. Fit Lab Quantizer and Set Task

Same `LabQuantizer` and `MIMIC4EHRMambaMortalityTask` as Section 2.
The task reads `hadm_id` (present in both MIMIC-III and MIMIC-IV) to
filter events per admission.

In [ ]:
abl_lab_quantizer = LabQuantizer(n_bins=5)
abl_lab_quantizer.fit_from_patients(list(mimic3_dataset.iter_patients()))

abl_task = MIMIC4EHRMambaMortalityTask(
    min_age=18,
    lab_quantizer=abl_lab_quantizer,
)
abl_sample_dataset = mimic3_dataset.set_task(abl_task)
abl_train_ds, abl_val_ds, abl_test_ds = split_by_sample(
    abl_sample_dataset, ratios=[0.5, 0.2, 0.3]
)

### 10. Create Data Loaders

Identical to Section 3 — `collate_ehr_mamba_batch` handles variable-length
sequences from MIMIC-III just as it does for MIMIC-IV.

In [ ]:
abl_train_loader = DataLoader(
    abl_train_ds, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
abl_val_loader = DataLoader(
    abl_val_ds, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
abl_test_loader = DataLoader(
    abl_test_ds, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)

### 11. Initialize EHRMamba Model

Same architecture as Section 4 — `use_ehr_mamba_embedding=True` activates
the full 7-component §2.2 embedding. The vocabulary is rebuilt from the
MIMIC-III sample dataset, so `word_embeddings` will have a different
vocab size than the MIMIC-IV model.

In [ ]:
abl_model = EHRMamba(
    dataset=abl_sample_dataset,
    embedding_dim=128,
    num_layers=2,
    state_size=16,
    conv_kernel=4,
    dropout=0.1,
    use_ehr_mamba_embedding=True,
)
abl_trainer = Trainer(model=abl_model, metrics=["roc_auc", "pr_auc"])
print(abl_model)

### 12. Train

In [ ]:
abl_trainer.train(
    train_dataloader=abl_train_loader,
    val_dataloader=abl_val_loader,
    epochs=50,
    monitor="roc_auc",
    optimizer_params={"lr": 1e-4},
)

### 13. Evaluate on Test Set

In [ ]:
abl_trainer.evaluate(abl_test_loader)


### 14. Get Patient Embeddings

Pooled patient representations from the MIMIC-III model — same API as
Section 7. Embedding shape will be `(B, 128)` regardless of dataset.

In [ ]:
abl_model.eval()
with torch.no_grad():
    abl_batch = next(iter(abl_test_loader))
    abl_output = abl_model(**abl_batch, embed=True)
    print(f"Patient embeddings shape: {abl_output['embed'].shape}")
    print(f"Predicted probabilities:  {abl_output['y_prob'].squeeze()}")
    print(f"Ground truth labels:      {abl_output['y_true'].squeeze()}")